<a href="https://colab.research.google.com/github/DhikshaSubash/market-sentiment-analysis/blob/main/notebooks/04_sentiment_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Install dependencies
!pip install transformers torch pyspark -q

print("✅ Libraries installed!")

In [ ]:
# Cell 2 - Load FinBERT from HuggingFace
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

print("⏳ Loading FinBERT model... (first time takes ~1 min)")

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# Labels FinBERT uses
labels = ["positive", "negative", "neutral"]

print("✅ FinBERT loaded successfully!")
print(f"Labels: {labels}")

In [ ]:
# Cell 3 - Test FinBERT on sample headlines
import torch.nn.functional as F

def get_sentiment(headline):
    inputs = tokenizer(
        headline,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)
    scores = probs[0].tolist()

    sentiment_scores = {
        "positive": round(scores[0], 4),
        "negative": round(scores[1], 4),
        "neutral":  round(scores[2], 4)
    }

    predicted_label = labels[scores.index(max(scores))]

    return predicted_label, sentiment_scores

# Test on sample headlines
test_headlines = [
    "Apple reports record breaking profits this quarter",
    "Tesla stock crashes after massive recall announcement",
    "Microsoft announces new product launch next month"
]

print("🧪 Testing FinBERT on sample headlines:\n")
for headline in test_headlines:
    label, scores = get_sentiment(headline)
    print(f"Headline : {headline}")
    print(f"Sentiment: {label.upper()}")
    print(f"Scores   : {scores}")
    print("-" * 60)

In [ ]:
# Cell 4 - Load processed news from Drive
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SentimentScoring") \
    .getOrCreate()

news_df = spark.read.parquet(
    "/content/drive/MyDrive/market_sentiment/processed/news"
)

print(f"✅ Loaded {news_df.count()} news records")
news_df.select("symbol", "headline").show(5, truncate=60)

In [ ]:
# Cell 5 - Score all headlines with FinBERT
import pandas as pd

# Convert to Pandas for FinBERT processing
news_pd = news_df.select("symbol", "headline", "published_at").toPandas()

print(f"⏳ Scoring {len(news_pd)} headlines with FinBERT...\n")

results = []
for idx, row in news_pd.iterrows():
    label, scores = get_sentiment(row['headline'])
    results.append({
        "symbol":         row['symbol'],
        "headline":       row['headline'],
        "published_at":   row['published_at'],
        "sentiment":      label,
        "positive_score": scores['positive'],
        "negative_score": scores['negative'],
        "neutral_score":  scores['neutral']
    })

    # Progress update every 10 headlines
    if (idx + 1) % 10 == 0:
        print(f"  Scored {idx + 1}/{len(news_pd)} headlines...")

print(f"\n✅ All {len(results)} headlines scored!")

In [ ]:
# Cell 6 - Convert results to Spark DataFrame
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType
)

sentiment_schema = StructType([
    StructField("symbol",         StringType(), True),
    StructField("headline",       StringType(), True),
    StructField("published_at",   StringType(), True),
    StructField("sentiment",      StringType(), True),
    StructField("positive_score", FloatType(),  True),
    StructField("negative_score", FloatType(),  True),
    StructField("neutral_score",  FloatType(),  True),
])

sentiment_df = spark.createDataFrame(results, schema=sentiment_schema)

print("✅ Sentiment Spark DataFrame created!")
print(f"\nSentiment distribution:")
sentiment_df.groupBy("sentiment").count().orderBy("count", ascending=False).show()

print("\nSample scored headlines:")
sentiment_df.select(
    "symbol", "headline", "sentiment", "positive_score", "negative_score"
).show(5, truncate=50)

In [ ]:
# Cell 7 - Sentiment summary per stock
from pyspark.sql.functions import round as spark_round, avg, count

summary_df = sentiment_df.groupBy("symbol").agg(
    count("headline").alias("total_articles"),
    spark_round(avg("positive_score"), 4).alias("avg_positive"),
    spark_round(avg("negative_score"), 4).alias("avg_negative"),
    spark_round(avg("neutral_score"),  4).alias("avg_neutral")
)

print("✅ Sentiment Summary Per Stock:\n")
summary_df.orderBy("avg_positive", ascending=False).show()

In [ ]:
# Cell 8 - Save sentiment scored data
sentiment_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/sentiment/scored_news"
)

summary_df.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/market_sentiment/sentiment/summary"
)

print("✅ Sentiment data saved to Drive!")
print("\nFolder structure so far:")
print("""
market_sentiment/
├── raw/
│   ├── news/        ✅
│   └── prices/      ✅
├── processed/
│   ├── news/        ✅
│   └── prices/      ✅
└── sentiment/
    ├── scored_news/ ✅
    └── summary/     ✅
""")